# Notebook 33 — Full Scaling Collapse of Phase-Locked CZ Quality

This notebook attempts the strongest synthesis in the series:

\[
S(\gamma, \gamma_\phi; T, \alpha, \omega_{\max})
\longrightarrow
\widehat S\!\left(\gamma_{\mathrm{eff}},\; T/T_c(\text{control})\right)
\]

where:
- `gamma_eff = gamma + lambda * gamma_phi`
- `T_c` is the extracted universality-boundary scale from the earlier notebooks.

Main goals:
1. reuse the effective noise coordinate,
2. reuse the extracted control-space boundary laws,
3. build a rescaled control variable `u = T / T_c`,
4. test whether multiple protocols collapse onto a shared response surface,
5. compare raw curves against the rescaled representation.

This is the direct follow-up to Notebook 32 and is the cleanest test of whether the project admits a true two-variable scaling form rather than separate partial reductions.


In [ ]:
# Ensure dependencies are installed before imports
import importlib, subprocess, sys

def ensure(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

for pkg in ['qutip', 'numpy', 'matplotlib', 'scipy']:
    ensure(pkg)


## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.optimize import minimize_scalar
from qutip import basis, qeye, tensor, sigmax, mesolve


## Basis states and operators

In [ ]:
g = basis(2, 0)
r = basis(2, 1)
I = qeye(2)

sx = sigmax()
n = r * r.dag()
sigma_minus = g * r.dag()

gg = tensor(g, g)
gr = tensor(g, r)
rg = tensor(r, g)
rr = tensor(r, r)

basis_states = [gg, gr, rg, rr]

sx1 = tensor(sx, I)
sx2 = tensor(I, sx)
n1 = tensor(n, I)
n2 = tensor(I, n)
sm1 = tensor(sigma_minus, I)
sm2 = tensor(I, sigma_minus)

U_cz = np.diag([1, 1, 1, -1]).astype(complex)


## Baseline protocol and selected protocol family

In [ ]:
baseline = {
    'T': 26.0,
    'alpha': 0.10,
    'omega_max': 1.09,
    'delta0': 1.00,
    'p': 1.00,
    'q': 0.72,
}
V = 4.0

T0 = baseline['T']
alpha0 = baseline['alpha']
omega0 = baseline['omega_max']

protocols = {
    'baseline': dict(baseline),
    'T_minus': {**baseline, 'T': 24.0},
    'T_plus': {**baseline, 'T': 30.0},
    'alpha_minus': {**baseline, 'alpha': 0.08},
    'alpha_plus': {**baseline, 'alpha': 0.12},
    'omega_minus': {**baseline, 'omega_max': 1.02},
    'omega_plus': {**baseline, 'omega_max': 1.16},
}

for name, pset in protocols.items():
    print(name, pset)


## Shaped schedules and Hamiltonian

In [ ]:
def omega_shaped(t, T, omega_max, p):
    s = t / T
    base = 16.0 * (s * (1.0 - s)) ** 2
    return omega_max * np.maximum(base, 0.0) ** p

def delta_shaped(t, T, V, alpha, delta0, q):
    s = t / T
    shaped = s ** q
    return delta0 + alpha * V * 0.5 * (1.0 - np.cos(np.pi * shaped))

H_omega = 0.5 * (sx1 + sx2)
H_delta = -(n1 + n2)
H_V = n1 * n2

def build_time_dependent_hamiltonian(T, omega_max, alpha, V, delta0, p, q):
    def coeff_omega(t, args=None):
        return omega_shaped(t, T=T, omega_max=omega_max, p=p)
    def coeff_delta(t, args=None):
        return delta_shaped(t, T=T, V=V, alpha=alpha, delta0=delta0, q=q)
    return [
        [H_omega, coeff_omega],
        [H_delta, coeff_delta],
        [H_V, lambda t, args=None: V],
    ]


## Collapse operators

In [ ]:
def collapse_ops(gamma_decay=0.0, gamma_phi=0.0):
    ops = []
    if gamma_decay > 0:
        ops.append(np.sqrt(gamma_decay) * sm1)
        ops.append(np.sqrt(gamma_decay) * sm2)
    if gamma_phi > 0:
        ops.append(np.sqrt(gamma_phi) * n1)
        ops.append(np.sqrt(gamma_phi) * n2)
    return ops


## Noisy evolution and effective map

In [ ]:
def run_noisy_evolution(psi0, params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    H = build_time_dependent_hamiltonian(
        params['T'], params['omega_max'], params['alpha'], V, params['delta0'], params['p'], params['q']
    )
    times = np.linspace(0.0, params['T'], n_steps)
    c_ops = collapse_ops(gamma_decay=gamma_decay, gamma_phi=gamma_phi)
    result = mesolve(H, psi0, times, c_ops)
    return result.states[-1]

def state_to_column_mixed(state):
    vals = []
    for basis_state in basis_states:
        if state.isket:
            vals.append(basis_state.overlap(state))
        else:
            val = basis_state.dag() * state * basis_state
            vals.append(np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val))
    return np.array(vals, dtype=complex)

def build_noisy_effective_map(params, gamma_decay=0.0, gamma_phi=0.0, n_steps=140):
    cols = []
    finals = []
    for psi0 in basis_states:
        final_state = run_noisy_evolution(
            psi0, params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=n_steps
        )
        cols.append(state_to_column_mixed(final_state))
        finals.append(final_state)
    return np.column_stack(cols), finals


## Diagnostics

In [ ]:
def process_fidelity(U_eff, U_target):
    d = U_target.shape[0]
    return float(np.abs(np.trace(np.conjugate(U_target.T) @ U_eff)) ** 2 / (d ** 2))

def wrapped_phase(x):
    return np.angle(np.exp(1j * x))

def diagonal_phases(U_eff):
    return np.angle(np.diag(U_eff))

def entangling_phase(U_eff):
    phi = diagonal_phases(U_eff)
    return wrapped_phase(phi[0] + phi[3] - phi[1] - phi[2])

def extract_local_z_phases(U_eff):
    phi00, phi01, phi10, phi11 = diagonal_phases(U_eff)
    phi_ent = entangling_phase(U_eff)
    a = wrapped_phase(phi10 - phi00)
    b = wrapped_phase(phi01 - phi00)
    global_phase = phi00
    return global_phase, a, b, phi_ent

def local_z_diagonal(a, b):
    return np.diag([1.0, np.exp(1j*b), np.exp(1j*a), np.exp(1j*(a+b))]).astype(complex)

def compensate_local_z(U_eff):
    global_phase, a, b, phi_ent = extract_local_z_phases(U_eff)
    U1 = np.exp(-1j * global_phase) * U_eff
    U2 = U1 @ local_z_diagonal(-a, -b)
    return U2, global_phase, a, b, phi_ent

def compensated_cz_fidelity(U_eff):
    U_comp, _, _, _, _ = compensate_local_z(U_eff)
    return process_fidelity(U_comp, U_cz)

def leakage_from_finals(final_states):
    scores = []
    for idx, final_state in enumerate(final_states):
        basis_state = basis_states[idx]
        if final_state.isket:
            score = np.abs(basis_state.overlap(final_state)) ** 2
        else:
            val = basis_state.dag() * final_state * basis_state
            score = np.real(val.full()[0, 0]) if hasattr(val, 'full') else np.real(val)
        scores.append(score)
    return float(1.0 - np.mean(scores))

def coherence_proxy(final_states):
    vals = []
    for state in final_states:
        rho = state.proj() if state.isket else state
        arr = rho.full()
        off = arr.copy()
        np.fill_diagonal(off, 0.0)
        vals.append(np.mean(np.abs(off)))
    return float(np.mean(vals))


## Noise grid

In [ ]:
gamma_decay_vals = np.linspace(0.0, 0.12, 13)
gamma_phi_vals = np.linspace(0.0, 0.12, 13)


## Evaluate one protocol

In [ ]:
def evaluate_protocol(params):
    cz_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    coh_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))
    leak_map = np.zeros((len(gamma_phi_vals), len(gamma_decay_vals)))

    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            U_eff, finals = build_noisy_effective_map(
                params, gamma_decay=gamma_decay, gamma_phi=gamma_phi, n_steps=140
            )
            cz_map[i, j] = compensated_cz_fidelity(U_eff)
            coh_map[i, j] = coherence_proxy(finals)
            leak_map[i, j] = leakage_from_finals(finals)

    S = cz_map * coh_map * (1.0 - leak_map)
    S_norm = S / np.max(S) if np.max(S) > 0 else S.copy()

    S_gamma = S_norm[0, :]
    S_phi = S_norm[:, 0]
    interp_gamma = interp1d(S_gamma * 0 + gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

    def loss(lam):
        mapped = lam * gamma_phi_vals
        pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
        return float(np.mean((pred - S_phi) ** 2))

    fit = minimize_scalar(loss, bounds=(0.1, 5.0), method='bounded')
    lam = float(fit.x)

    gamma_eff_grid = np.zeros_like(S_norm)
    for i, gamma_phi in enumerate(gamma_phi_vals):
        for j, gamma_decay in enumerate(gamma_decay_vals):
            gamma_eff_grid[i, j] = gamma_decay + lam * gamma_phi

    return {
        'params': params,
        'S_norm': S_norm,
        'lambda': lam,
        'gamma_eff_grid': gamma_eff_grid,
    }


## Baseline reference and boundary-law fitting

In [ ]:
def build_boundary_laws():
    baseline_res = evaluate_protocol(baseline)
    lambda0 = baseline_res['lambda']

    def score_for_params(params):
        res = evaluate_protocol(params)
        S = res['S_norm']
        S_gamma = S[0, :]
        S_phi = S[:, 0]
        interp_gamma = interp1d(gamma_decay_vals, S_gamma, kind='linear', fill_value='extrapolate')

        def axis_loss(lam):
            mapped = lam * gamma_phi_vals
            pred = interp_gamma(np.clip(mapped, gamma_decay_vals.min(), gamma_decay_vals.max()))
            return float(np.mean((pred - S_phi) ** 2))

        aloss = axis_loss(res['lambda'])

        # curve mismatch against baseline
        bS = baseline_res['S_norm']
        bge = baseline_res['gamma_eff_grid'].ravel()
        bsv = bS.ravel()
        order = np.argsort(bge)
        bge = bge[order]
        bsv = bsv[order]
        bins = np.linspace(bge.min(), bge.max(), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        bmeans = np.full(len(centers), np.nan)
        for k in range(len(centers)):
            mask = (bge >= bins[k]) & (bge < bins[k+1])
            if np.any(mask):
                bmeans[k] = np.mean(bsv[mask])
        bvalid = ~np.isnan(bmeans)
        interp0 = interp1d(centers[bvalid], bmeans[bvalid], kind='linear', fill_value='extrapolate')

        ge = res['gamma_eff_grid'].ravel()
        sv = S.ravel()
        order = np.argsort(ge)
        ge = ge[order]
        sv = sv[order]
        bins = np.linspace(ge.min(), ge.max(), 13)
        centers = 0.5 * (bins[:-1] + bins[1:])
        means = np.full(len(centers), np.nan)
        for k in range(len(centers)):
            mask = (ge >= bins[k]) & (ge < bins[k+1])
            if np.any(mask):
                means[k] = np.mean(sv[mask])
        valid = ~np.isnan(means)
        curve_mismatch = float(np.mean((means[valid] - interp0(np.clip(centers[valid], centers[bvalid].min(), centers[bvalid].max()))) ** 2))
        lambda_shift = abs(res['lambda'] - lambda0)

        return float(np.exp(
            -(lambda_shift / 0.15)
            -(aloss / 0.001)
            -(curve_mismatch / 0.001)
        ))

    boundary_level = 0.30
    T_vals_local = np.array([20.0, 24.0, 26.0, 30.0, 34.0])
    alpha_vals_local = np.array([0.06, 0.08, 0.10, 0.12, 0.14])
    omega_vals_local = np.array([0.95, 1.02, 1.09, 1.16, 1.23])

    TA_score = np.zeros((len(alpha_vals_local), len(T_vals_local)))
    TO_score = np.zeros((len(omega_vals_local), len(T_vals_local)))

    for i, alpha in enumerate(alpha_vals_local):
        for j, T in enumerate(T_vals_local):
            TA_score[i, j] = score_for_params({**baseline, 'T': float(T), 'alpha': float(alpha)})

    for i, omega in enumerate(omega_vals_local):
        for j, T in enumerate(T_vals_local):
            TO_score[i, j] = score_for_params({**baseline, 'T': float(T), 'omega_max': float(omega)})

    def extract_vertical_boundary(score_map, xvals, yvals, level):
        bx, by = [], []
        for i, y in enumerate(yvals):
            row = score_map[i, :]
            crossing = None
            for j in range(len(xvals) - 1):
                v0, v1 = row[j], row[j + 1]
                if (v0 >= level and v1 < level) or (v0 > level and v1 <= level):
                    x0, x1 = xvals[j], xvals[j + 1]
                    xc = x0 if v1 == v0 else x0 + (level - v0) * (x1 - x0) / (v1 - v0)
                    crossing = xc
                    break
            if crossing is None and np.max(row) >= level and np.min(row) >= level:
                crossing = float(xvals[-1])
            if crossing is not None:
                bx.append(float(crossing))
                by.append(float(y))
        return np.array(bx), np.array(by)

    TA_bx, TA_by = extract_vertical_boundary(TA_score, T_vals_local, alpha_vals_local, boundary_level)
    TO_bx, TO_by = extract_vertical_boundary(TO_score, T_vals_local, omega_vals_local, boundary_level)

    TA_x = TA_by / alpha0
    TA_y = TA_bx / T0
    TO_x = TO_by / omega0
    TO_y = TO_bx / T0

    coeff_alpha = np.polyfit(TA_x - 1.0, TA_y, deg=1)
    coeff_omega = np.polyfit(TO_x, TO_y, deg=1)

    return coeff_alpha, coeff_omega, TA_x, TA_y, TO_x, TO_y

coeff_alpha, coeff_omega, TA_x, TA_y, TO_x, TO_y = build_boundary_laws()
print("alpha shifted-linear coeffs:", coeff_alpha)
print("omega linear coeffs:", coeff_omega)


## Define scaling variables

In [ ]:
def Tc_over_T0_alpha(alpha):
    x = alpha / alpha0
    return np.polyval(coeff_alpha, x - 1.0)

def Tc_over_T0_omega(omega):
    x = omega / omega0
    return np.polyval(coeff_omega, x)

def Tc_from_params(params):
    Tc_alpha = T0 * Tc_over_T0_alpha(params['alpha'])
    Tc_omega = T0 * Tc_over_T0_omega(params['omega_max'])
    return float(np.sqrt(np.maximum(Tc_alpha, 1e-9) * np.maximum(Tc_omega, 1e-9)))

def control_scale_u(params):
    return float(params['T'] / Tc_from_params(params))

for name, pset in protocols.items():
    print(name, "Tc ≈", Tc_from_params(pset), "u = T/Tc ≈", control_scale_u(pset))


## Evaluate protocol family

In [ ]:
family = {}
for name, pset in protocols.items():
    res = evaluate_protocol(pset)
    res['u'] = control_scale_u(pset)
    family[name] = res
    print(name, "lambda =", res['lambda'], "u =", res['u'])


## Raw response curves by protocol

In [ ]:
plt.figure(figsize=(7.6, 5.0))
for name, res in family.items():
    ge = res['gamma_eff_grid'].ravel()
    sv = res['S_norm'].ravel()
    order = np.argsort(ge)
    ge = ge[order]
    sv = sv[order]
    bins = np.linspace(ge.min(), ge.max(), 14)
    centers = 0.5 * (bins[:-1] + bins[:-1] + np.diff(bins))
    xs = []
    means = []
    for k in range(len(bins) - 1):
        mask = (ge >= bins[k]) & (ge < bins[k+1])
        if np.any(mask):
            xs.append(centers[k])
            means.append(np.mean(sv[mask]))
    plt.plot(xs, means, label=name)

plt.xlabel('gamma_eff')
plt.ylabel('Normalized S')
plt.title('Raw response curves before control rescaling')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Plot the full scaling collapse

In [ ]:
plt.figure(figsize=(7.8, 5.4))
for name, res in family.items():
    u = res['u']
    ge = res['gamma_eff_grid'].ravel()
    sv = res['S_norm'].ravel()
    x_scaled = ge * u

    order = np.argsort(x_scaled)
    x_sorted = x_scaled[order]
    s_sorted = sv[order]

    bins = np.linspace(x_sorted.min(), x_sorted.max(), 14)
    centers = 0.5 * (bins[:-1] + bins[:-1] + np.diff(bins))
    xs = []
    means = []
    for k in range(len(bins) - 1):
        mask = (x_sorted >= bins[k]) & (x_sorted < bins[k+1])
        if np.any(mask):
            xs.append(centers[k])
            means.append(np.mean(s_sorted[mask]))

    plt.plot(xs, means, label=f"{name} (u={u:.3f})")

plt.xlabel('scaled coordinate: gamma_eff * (T/Tc)')
plt.ylabel('Normalized S')
plt.title('Attempted full scaling collapse')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## Shared collapsed predictor

In [ ]:
all_x = []
all_s = []

for name, res in family.items():
    u = res['u']
    ge = res['gamma_eff_grid'].ravel()
    sv = res['S_norm'].ravel()
    all_x.extend(list(ge * u))
    all_s.extend(list(sv))

all_x = np.array(all_x, dtype=float)
all_s = np.array(all_s, dtype=float)

order = np.argsort(all_x)
all_x = all_x[order]
all_s = all_s[order]

bins = np.linspace(all_x.min(), all_x.max(), 18)
centers = 0.5 * (bins[:-1] + bins[:-1] + np.diff(bins))
means = np.full(len(bins) - 1, np.nan)

for k in range(len(bins) - 1):
    mask = (all_x >= bins[k]) & (all_x < bins[k+1])
    if np.any(mask):
        means[k] = np.mean(all_s[mask])

valid = ~np.isnan(means)
collapse_interp = interp1d(centers[valid], means[valid], kind='linear', fill_value='extrapolate')


## Compare each protocol to the shared collapsed curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for name, res in family.items():
    u = res['u']
    ge = res['gamma_eff_grid'].ravel()
    sv = res['S_norm'].ravel()
    x_scaled = ge * u

    order = np.argsort(x_scaled)
    x_sorted = x_scaled[order]
    s_sorted = sv[order]

    bins = np.linspace(x_sorted.min(), x_sorted.max(), 14)
    centers_p = 0.5 * (bins[:-1] + bins[:-1] + np.diff(bins))
    xs = []
    means_p = []
    for k in range(len(bins) - 1):
        mask = (x_sorted >= bins[k]) & (x_sorted < bins[k+1])
        if np.any(mask):
            xs.append(centers_p[k])
            means_p.append(np.mean(s_sorted[mask]))
    xs = np.array(xs)
    means_p = np.array(means_p)

    pred = collapse_interp(np.clip(xs, centers[valid].min(), centers[valid].max()))
    resid = means_p - pred

    axes[0].plot(xs, means_p, label=name)
    axes[1].plot(xs, resid, label=name)

axes[0].plot(centers[valid], means[valid], color='black', linewidth=2.4, label='shared collapsed curve')
axes[0].set_xlabel('gamma_eff * (T/Tc)')
axes[0].set_ylabel('Normalized S')
axes[0].set_title('Collapsed curves vs shared predictor')
axes[0].legend(fontsize=8)

axes[1].axhline(0.0, linestyle='--', alpha=0.7)
axes[1].set_xlabel('gamma_eff * (T/Tc)')
axes[1].set_ylabel('residual')
axes[1].set_title('Residuals to shared collapsed curve')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## Collapse-quality summary

In [ ]:
collapse_metrics = {}

for name, res in family.items():
    u = res['u']
    ge = res['gamma_eff_grid'].ravel()
    sv = res['S_norm'].ravel()
    x_scaled = ge * u

    order = np.argsort(x_scaled)
    x_sorted = x_scaled[order]
    s_sorted = sv[order]

    bins = np.linspace(x_sorted.min(), x_sorted.max(), 14)
    centers_p = 0.5 * (bins[:-1] + bins[:-1] + np.diff(bins))
    xs = []
    means_p = []
    for k in range(len(bins) - 1):
        mask = (x_sorted >= bins[k]) & (x_sorted < bins[k+1])
        if np.any(mask):
            xs.append(centers_p[k])
            means_p.append(np.mean(s_sorted[mask]))
    xs = np.array(xs)
    means_p = np.array(means_p)

    pred = collapse_interp(np.clip(xs, centers[valid].min(), centers[valid].max()))
    mse = float(np.mean((means_p - pred) ** 2))
    mae = float(np.mean(np.abs(means_p - pred)))
    collapse_metrics[name] = {'u': u, 'mse': mse, 'mae': mae}
    print(name, collapse_metrics[name])


## Compact summary

In [ ]:
lines = []
lines.append("Full scaling collapse summary")
lines.append("")
lines.append("Boundary-law coefficients:")
lines.append(f"- alpha shifted-linear coeffs = {coeff_alpha}")
lines.append(f"- omega linear coeffs = {coeff_omega}")
lines.append("")

lines.append("Control rescaling:")
for name, res in family.items():
    lines.append(
        f"- {name}: T={res['params']['T']:.3f}, alpha={res['params']['alpha']:.3f}, "
        f"omega={res['params']['omega_max']:.3f}, u=T/Tc={res['u']:.4f}"
    )

lines.append("")
lines.append("Collapse-quality metrics against shared predictor:")
for name, met in collapse_metrics.items():
    lines.append(f"- {name}: MSE={met['mse']:.6e}, MAE={met['mae']:.6e}")

lines.append("")
lines.append("Interpretation:")
lines.append("- If the collapsed curves align, then the gate-quality landscape admits a two-variable scaling form.")
lines.append("- gamma_eff handles the noise-space reduction.")
lines.append("- T/Tc handles the leading control-space deformation.")
lines.append("- Residual structure indicates where additional variables still matter.")

summary_text = "\n".join(lines)
print(summary_text)


## Final conclusion

This notebook tests the strongest synthesis in the project:

\[
S(\gamma, \gamma_\phi; T, \alpha, \omega_{\max})
\approx
\widehat S\!\left(\gamma_{\mathrm{eff}}\, T/T_c\right)
\]

with:
- `gamma_eff = gamma + lambda * gamma_phi`
- `T_c` extracted from the control-space universality boundary.

If the curves align well under this rescaling, then the project reaches a genuine scaling-collapse statement:
the phase-locked CZ family is organized by a reduced noise coordinate together with a reduced control coordinate.
